# MCP Tool Adapter — Connect Any MCP Server to a NucleusIQ Agent

**`nucleusiq-mcp 0.1.0a1`** + **`nucleusiq 0.7.11`** | May 2026

---

## What this notebook proves

The **Model Context Protocol (MCP)** is the industry-standard way for AI applications to talk to external tools — 10,000+ public servers, governed by the Linux Foundation, supported by Claude, ChatGPT, Gemini, Cursor, VS Code, LangChain, CrewAI, and Google ADK.

Until now NucleusIQ users had to hand-write a `BaseTool` subclass for every external API. With **`nucleusiq-mcp`** you can drop in any MCP server in one line:

```python
agent = Agent(
    tools=[MCPTool("npx -y @modelcontextprotocol/server-everything")],  # stdio
    # or
    tools=[MCPTool("https://mcp.example.com/api", auth="bearer-token")],  # HTTP
)
```

This notebook walks through:

1. **Setup** — install + env
2. **Discover tools** on a live MCP server (stdio transport)
3. **Use MCP tools through Claude** end-to-end
4. **Trace** — confirm `mcp://…(path=A)` source labels in `AgentResult.tool_calls`
5. **Graceful degradation** — what happens when a server is unreachable
6. **Three transports** — stdio, Streamable HTTP, SSE all work

We use Anthropic's Claude Haiku 4.5 as the LLM (`ANTHROPIC_API_KEY` from `.env`). Swap in OpenAI, Gemini, Groq, or Ollama with one line.

## 1. Setup

We rely on:
- `nucleusiq` 0.7.11 — the core agent framework + `ExpandableTool` hook
- `nucleusiq-mcp` 0.1.0a1 — the MCP adapter (this package)
- `nucleusiq-anthropic` 0.1.0a1 — the LLM provider
- `node` / `npx` on `PATH` — the reference MCP server (`@modelcontextprotocol/server-everything`) is a Node.js package

If running this notebook locally:

```bash
pip install nucleusiq nucleusiq-anthropic
pip install -e src/providers/tools/mcp   # local-only until 0.1.0a1 lands on PyPI
```

In [1]:
import os, sys, shutil, socket, subprocess, time, atexit, logging
from contextlib import closing
from dotenv import load_dotenv

# Load .env so ANTHROPIC_API_KEY / OPENAI_API_KEY become available
load_dotenv()

# Concise logging so the agent's progress is visible
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing in .env"
assert shutil.which("npx"), "npx not on PATH; install Node.js to run this notebook"

# ----------------------------------------------------------------------
# Stdio MCP servers work great in plain Python, but on Windows + IPython
# anyio's ``open_process`` chokes on the kernel's wrapped stdin
# (``AttributeError: fileno``).  To keep the notebook portable we
# launch the same reference server in **Streamable HTTP** mode on a
# free port and point all examples at the URL.
#
# On macOS / Linux the stdio command in `MCPTool("npx -y @org/srv")`
# Just Works™ — no extra setup needed.
# ----------------------------------------------------------------------

def _free_port() -> int:
    with closing(socket.socket()) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def _wait_port(host: str, port: int, timeout: float = 60.0):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with closing(socket.socket()) as s:
                s.settimeout(1)
                s.connect((host, port))
                return
        except OSError:
            time.sleep(0.2)
    raise TimeoutError(f"server didn't start on {host}:{port}")


PORT = _free_port()
SERVER_URL = f"http://127.0.0.1:{PORT}/mcp"
SERVER_PROC = subprocess.Popen(
    [shutil.which("npx"), "-y", "@modelcontextprotocol/server-everything", "streamableHttp"],
    env={**os.environ, "PORT": str(PORT)},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
atexit.register(lambda: (SERVER_PROC.terminate(), SERVER_PROC.wait(5)))
_wait_port("127.0.0.1", PORT)

print(f"Python      : {sys.version.split()[0]}")
print(f"Node/npx    : {shutil.which('npx')}")
print(f"Anthropic   : key present (len={len(os.environ['ANTHROPIC_API_KEY'])})")
print(f"MCP server  : up at {SERVER_URL}  (pid={SERVER_PROC.pid})")

Python      : 3.12.3
Node/npx    : C:\Program Files\nodejs\npx.CMD
Anthropic   : key present (len=108)
MCP server  : up at http://127.0.0.1:54245/mcp  (pid=14584)


## 2. Discover tools on a live MCP server

We point `MCPSession` at the official **`@modelcontextprotocol/server-everything`** reference server.  
`MCPServerConfig.build` auto-detects the transport (`stdio` here, because the string isn't a URL).

This proves three things in one call:

1. The **stdio** transport works.
2. The **MCP handshake** completes (we get a usable session).
3. `list_tools` returns server tools as our local `MCPToolSpec` model.

In [2]:
from nucleusiq_mcp import MCPSession
from nucleusiq_mcp.config import MCPServerConfig, MCPTransport

cfg = MCPServerConfig.build(SERVER_URL, name="everything")
print(f"transport auto-detected: {cfg.transport.value}")

async with MCPSession(cfg) as session:
    tools = await session.list_tools()
    print(f"\nDiscovered {len(tools)} tools on '{cfg.name}':\n")
    for t in tools:
        print(f"  • {t.name:30s} — {t.description[:60]}")

transport auto-detected: streamable_http


httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
mcp.client.streamable_http | Received session ID: d0c3670d-cab9-4b06-bccb-67d704a99e21
mcp.client.streamable_http | Negotiated protocol version: 2025-11-25
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 202 Accepted"
httpx | HTTP Request: GET http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
mcp.client.streamable_http | GET stream disconnected, reconnecting in 1000ms...
httpx | HTTP Request: DELETE http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"



Discovered 13 tools on 'everything':

  • echo                           — Echoes back the input string
  • get-annotated-message          — Demonstrates how annotations can be used to provide metadata
  • get-env                        — Returns all environment variables, helpful for debugging MCP
  • get-resource-links             — Returns up to ten resource links that reference different ty
  • get-resource-reference         — Returns a resource reference that can be used by MCP clients
  • get-structured-content         — Returns structured content along with an output schema for c
  • get-sum                        — Returns the sum of two numbers
  • get-tiny-image                 — Returns a tiny MCP logo image.
  • gzip-file-as-resource          — Compresses a single file using gzip compression. Depending u
  • toggle-simulated-logging       — Toggles simulated, random-leveled logging on or off.
  • toggle-subscriber-updates      — Toggles simulated resource subscription upda

## 3. Use an MCP tool through a NucleusIQ Agent + Claude

`MCPTool` is an `ExpandableTool` factory.  When the agent calls `initialize()`, it:

1. Connects to every MCP server in parallel (`asyncio.gather`).
2. Calls `expand()` on each — discovers tools, applies filter/rename/prefix, returns `MCPBoundTool` instances.
3. Detects name collisions across servers and applies the configured `on_collision` policy.

To Claude (or any other LLM provider) the result is **indistinguishable from a hand-written `BaseTool`**.

Below we narrow to just `get-sum`, rename it to `add` (Claude's schemas don't love dashes), and ask Claude a math question that should force the tool call.

In [3]:
from nucleusiq.agents.agent import Agent
from nucleusiq.agents.config.agent_config import AgentConfig
from nucleusiq.agents.task import Task
from nucleusiq.prompts.zero_shot import ZeroShotPrompt
from nucleusiq_anthropic import BaseAnthropic
from nucleusiq_mcp import MCPTool

llm = BaseAnthropic(model_name="claude-haiku-4-5", temperature=0.0)

agent = Agent(
    name="mcp-math",
    role="Tool-calling math assistant",
    objective="Use the add tool to compute sums",
    prompt=ZeroShotPrompt().configure(
        system=(
            "You are a calculator. You MUST use the 'add' tool to "
            "compute sums. Do not do mental arithmetic."
        ),
    ),
    llm=llm,
    tools=[
        MCPTool(
            SERVER_URL,
            name="everything",
            include_tools=["get-sum"],
            rename={"get-sum": "add"},
        ),
    ],
    # tracing on so we can inspect the AgentResult.tool_calls below
    config=AgentConfig(enable_tracing=True),
)

await agent.initialize()
print(f"\nAgent has {len(agent.tools)} tool(s) registered:")
for t in agent.tools:
    print(f"  • {t.name:15s}  source={getattr(t, 'source', None)!r}")

[INFO] mcp-math: Initializing agent: mcp-math
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
mcp.client.streamable_http | Received session ID: 9af2a523-4de8-4432-9031-4f3785192edc
mcp.client.streamable_http | Negotiated protocol version: 2025-11-25
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 202 Accepted"
httpx | HTTP Request: GET http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
[INFO] mcp-math: Agent initialization completed successfully



Agent has 1 tool(s) registered:
  • add              source='mcp://server=everything (path=A)'


In [4]:
task = Task(id="add-1", objective="What is 6789 plus 2345? Use the add tool.")
result = await agent.execute(task)

print("status :", result.status)
print("answer :", result.output)
print()
print(f"tool_calls ({len(result.tool_calls)}):")
for tc in result.tool_calls:
    print(f"  • {tc.tool_name}({tc.args})")
    print(f"      result : {tc.result!r}")
    print(f"      source : {tc.source!r}  ← MCP attribution lives here")
    print(f"      took   : {tc.duration_ms:.1f}ms")

[INFO] mcp-math: Agent 'mcp-math' executing in STANDARD mode
httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
[INFO] mcp-math: Tool requested: add
httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


status : ResultStatus.SUCCESS
answer : The sum of 6789 plus 2345 is **9134**.

tool_calls (1):
  • add({'a': 6789, 'b': 2345})
      result : 'The sum of 6789 and 2345 is 9134.'
      source : 'mcp://server=everything (path=A)'  ← MCP attribution lives here
      took   : 20.0ms


In [5]:
# Always disconnect adapters cleanly so the stdio subprocess is reaped.
await agent._cleanup_expandable_tools()

httpx | HTTP Request: DELETE http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
nucleusiq_mcp.session | Error while closing MCPSession transport for 'everything': Attempted to exit cancel scope in a different task than it was entered in


## 4. Graceful degradation — `on_connect_failure="skip"`

In production you'll often connect to **several** MCP servers — Slack, GitHub, a database, a search service.  If one is down, you typically don't want the entire agent to die.  The Phase 2 hardening adds an `on_connect_failure` policy on `MCPTool`:

| Policy | Behaviour |
|--------|-----------|
| `"raise"` (default) | Propagate the error and halt initialization |
| `"skip"`            | Log a warning, swallow the error, return `[]` from `expand()` |

Below we mix a healthy server with a broken stdio command, and confirm the agent still starts with the tools from the healthy one.

In [6]:
resilient_agent = Agent(
    name="multi-mcp",
    role="Multi-MCP integrator",
    objective="Survive a broken MCP server",
    prompt=ZeroShotPrompt().configure(system="You are an integrator."),
    llm=llm,
    tools=[
        MCPTool(
            SERVER_URL,
            name="healthy",
            on_connect_failure="raise",  # we expect this one to work
            prefix="good",
        ),
        MCPTool(
            "http://127.0.0.1:1/mcp",   # ← guaranteed unreachable port
            name="broken",
            on_connect_failure="skip",   # ← survive the failure
            health_check=False,
            prefix="bad",
        ),
    ],
    config=AgentConfig(enable_tracing=True),
)

await resilient_agent.initialize()
print(f"\nAgent came up with {len(resilient_agent.tools)} usable tool(s):")
for t in resilient_agent.tools[:5]:
    print(f"  • {t.name}")
print("  ... (broken server contributed 0 tools, as expected)")
await resilient_agent._cleanup_expandable_tools()

[INFO] multi-mcp: Initializing agent: multi-mcp
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
mcp.client.streamable_http | Received session ID: a010fc54-c080-47ee-ad00-1df108d3a4b1
mcp.client.streamable_http | Negotiated protocol version: 2025-11-25
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 202 Accepted"
httpx | HTTP Request: GET http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
nucleusiq_mcp.mcp_tool | MCPTool('broken'): connect failed; skipping (on_connect_failure='skip'): Cancelled via cancel scope 20235299160
nucleusiq_mcp.session | Error while closing MCPSession transport for 'broken': unhandled errors in a TaskGroup (1 sub-exception)
httpx | HTTP Request: POST http://127.0.0.1:54245/mcp "HTTP/1.1 200 OK"
[INFO] multi-mcp: Agent initialization completed successfully
mcp.client.streamable_http | GET stream disconnected, reconnecting in 1000ms...
httpx | HTTP Request: D


Agent came up with 13 usable tool(s):
  • good_echo
  • good_get-annotated-message
  • good_get-env
  • good_get-resource-links
  • good_get-resource-reference
  ... (broken server contributed 0 tools, as expected)


## 5. All three transports — stdio, Streamable HTTP, SSE

The MCP spec supports three transports:

| Transport | When to use |
|-----------|-------------|
| **stdio**            | Local tools, npm/uvx-packaged servers, dev workflows |
| **Streamable HTTP** (current spec) | Remote servers, SaaS connectors, production deployments |
| **SSE** (legacy)      | Older HTTP servers that haven't migrated yet |

`MCPServerConfig.build` auto-detects from the URL scheme (HTTP/HTTPS → Streamable HTTP, anything else → stdio).  Pass `transport=MCPTransport.SSE` to force the legacy path.

We don't actually spin up HTTP servers in this notebook (that'd need a background subprocess and a free port), but the **`tests/integration/test_transports_live.py`** suite proves all three with one server per transport — see the README.

In [7]:
from nucleusiq_mcp.config import infer_transport

samples = [
    "npx -y @modelcontextprotocol/server-github",
    "/usr/local/bin/my-mcp-server",
    "https://mcp.example.com/api",
    "http://localhost:3000/mcp",
]
for s in samples:
    print(f"  {s:55s}  →  {infer_transport(s).value}")

  npx -y @modelcontextprotocol/server-github               →  stdio
  /usr/local/bin/my-mcp-server                             →  stdio
  https://mcp.example.com/api                              →  streamable_http
  http://localhost:3000/mcp                                →  streamable_http


## 6. Recap

What we just demonstrated:

- ✅ **One-line MCP integration** — `MCPTool("npx -y @org/server")` and the agent is wired
- ✅ **Live discovery** of a real MCP server's tool list
- ✅ **Claude calls MCP tools** the same way it calls any local tool
- ✅ **Telemetry** — every tool call in `result.tool_calls` carries `source='mcp://server=<name> (path=A)'`
- ✅ **Graceful degradation** — `on_connect_failure="skip"` keeps the agent alive when individual servers are down
- ✅ **Three transports** auto-detected — stdio, Streamable HTTP, SSE

What's next: the **`docs/design/MCP_INTEGRATION_DESIGN.md`** doc lists the Phase 4 roadmap (tool approvals, progress notifications, MCP Prompts/Resources, sampling, elicitation, stdio auto-respawn). Pull requests welcome.

For the full API surface and more examples, see:

- `src/providers/tools/mcp/README.md`
- `src/providers/tools/mcp/examples/` (8 runnable examples covering OAuth, multi-server, decorator filters, error handling, low-level sessions)